# NumPy Indexing, Views, Broadcasting, and Reshaping

In [1]:
import numpy as np

In [2]:
# Basic slicing (views)

a1 = np.arange(10)

print(a1[2:5]) # elements 2,3,4
print(a1[:5]) # first 5 elements
print(a1[:2]) # first 2 elements
print(a1[::2]) # step size 2

[2 3 4]
[0 1 2 3 4]
[0 1]
[0 2 4 6 8]


In [3]:
# Note: slicing gives us a view, not a copy.

x = a1[2:5]
x[:] = 99

print(a1) # original array changed

[ 0  1 99 99 99  5  6  7  8  9]


In [4]:
# This happens because x shares memory with a1.
np.shares_memory(a1, x)

True

In [5]:
# 2D Indexing

a2 = np.arange(12).reshape(3,4)

print(a2)
print(a2[1,:]) # row 1
print(a2[:,2]) # column 2
print(a2[0:2,1:3]) # submatrix

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[4 5 6 7]
[ 2  6 10]
[[1 2]
 [5 6]]


In [6]:
# Fancy indexing uses integer arrays or lists.

a3 = np.array([10,20,30,40,50])
a3[[0,2,4]]

array([10, 30, 50])

In [7]:
a2[[0, 2], [1, 3]] 

array([ 1, 11])

In [8]:
# fancy indexing gives us a view, not a copy.

x = a3[[0,2]]
x[:] = 99

print(a3) # original NOT changed
print(x)

# safer but more memory-expensive.
np.shares_memory(a3, x)

[10 20 30 40 50]
[99 99]


False

In [9]:
# Boolean masking uses a boolean array as a mask (of the same shape).

a4 = np.arange(1,7)
mask = a4 > 3

print(a4[mask]) # selection returns copy

[4 5 6]


In [10]:
print(a4[a4>3])
print(a4)

[4 5 6]
[1 2 3 4 5 6]


In [11]:
# Selection returns a copy, but assignment affects the original.

a4[a4>3]=0
print(a4)

[1 2 3 0 0 0]


In [12]:
# We can force the slicing to use copy as it's practission.

a5 = np.arange(6)
x = a5[1:4].copy()
x[:] = 0

print(x)
print(a5) # unchanged

[0 0 0]
[0 1 2 3 4 5]


Testing some of the functions

In [13]:
# Reshape returns a view, not a copy.

a6 = np.arange(12)
a6_rshped = a6.reshape(3,4)
a6 *= 2

print(a6_rshped)

[[ 0  2  4  6]
 [ 8 10 12 14]
 [16 18 20 22]]


In [14]:
# Ravel makes an array flat to 1D, and returns a view if possible, otherwise a copy.
# It is faster and more memory-efficient than flatten.

a7 = np.array([[1,2],[3,4]])
print(a7)

a7_rvled = a7.ravel()
a7+=2
print(a7_rvled)

[[1 2]
 [3 4]]
[3 4 5 6]


In [15]:
# Flatten makes an array flat to 1D, and always returns a copy.

a8 = np.array([[1,2],[3,4]])
a8_flted = a8.flatten()
a8-=11

print(a8_flted) # unchanged
print(a8)

[1 2 3 4]
[[-10  -9]
 [ -8  -7]]


In [16]:
# Transpose reverses or permutes axes, and returns a view, not a copy.
# Note: use in EEG/MEG for reordering axes.

a9 = np.array([[1,2],[3,4]])

print("Transposed a9:\n", a9.transpose()) # or a9.T
print("Axes swaped a9:\n", a9.swapaxes(0,1)) # swapaxes exchanges exactly two axes.

Transposed a9:
 [[1 3]
 [2 4]]
Axes swaped a9:
 [[1 3]
 [2 4]]


Vstack, hstack, concatenate: combine arrays along specified dimensions.

In [17]:
# Vstack stacks arrays row-wise (axis = 0 after forcing 2D).
np.vstack([[1,2], [3,4],[5,6]])

array([[1, 2],
       [3, 4],
       [5, 6]])

In [18]:
# Hstack stacks arrays column-wise (axis = 1 for 2D).
print(np.hstack([[[1,2],[3,4]],[[5,6],[7,8]]]))

# Hstack behaves differently for 1D arrays (it concatenates, not column-stacks).
np.hstack([1,2,[3,4],[5,6]])

[[1 2 5 6]
 [3 4 7 8]]


array([1, 2, 3, 4, 5, 6])

In [19]:
np.concatenate([[1,2], [3,4], [5,6]], axis=0)
# All dimensions except the concatenation axis must match.

array([1, 2, 3, 4, 5, 6])

In [20]:
# Let's practice with simulating subject-trial matrices.

n_subjects = 10
n_trials = 10
rng = np.random.default_rng(seed=42)

subjects = []
for _ in range(n_subjects):
    rt = rng.normal(loc=500, scale=80, size=n_trials)
    subjects.append(rt)

# Convert to a 2D matrix:
data = np.vstack(subjects)

# Now rows are subjects, and columns are trials.

In [21]:
# Let's build another mini-dataset and stack it with our data.

accuracy = []
for _ in range(n_subjects):
    acc = rng.binomial(n=1, p=0.85, size=n_trials)
    accuracy.append(acc)

accuracy = np.vstack(accuracy)
# Now we have data (reaction times) and accuracy (correctness), and both have a 10*10 shape.

# Stack them together:
group_data = np.stack([data, accuracy], axis=2)
print("group_data shape: ", group_data.shape)

# And reshape the final dataset:
group_data_rshped = group_data.reshape(-1,2)
# Now each row has one trial from one subject.
print("Long group_data shape: ", group_data_rshped.shape)

group_data shape:  (10, 10, 2)
Long group_data shape:  (100, 2)


Broadcasting Rulls
1. Compare the shapes of the two arrays from right to left.
2. Two dimensions are compatible if:
    - They are equal
    - One of them is 1
4. If shapes are not compatible, broadcasting fails.

In [22]:
a10 = np.array([1,2,3])
a11 = np.array([[10],[20]])
a10_11 = a10 + a11

print(a10_11)

[[11 12 13]
 [21 22 23]]


scalar: expanded to match any array shape.\
vector: can broadcast across matching dimensions or trailing dimension of matrix.\
matrix: only broadcasts if other array’s shape is compatible.

In [23]:
print(a10 + 10) # Scalar 10 broadcasts to shape (3,)

# Vector + Matrix
mat = np.array([[1,2,3],[4,5,6]])
vec = np.array([10,20,30])
print(mat + vec) # vec broadcasts along rows

[11 12 13]
[[11 22 33]
 [14 25 36]]


In [24]:
# Break
a12 = np.array([[1,2],[3,4]])
# a12+a10 will result in an error.

In [25]:
# Let's manually compute a normalization, then vectorize it.
# Normalization formula: x_norm = (x - min(x)) / (max(x) - min (x))

x = np.array([1,3,5,7])
x_max = x.max()
x_min = x.min()
x_norm_manual = [(xi - x_min)/(x_max-x_min) for xi in x]

print(x_norm_manual)

x_norm_vec = (x - x.min()) / (x.max() - x.min())

print(x_norm_vec)

[np.float64(0.0), np.float64(0.3333333333333333), np.float64(0.6666666666666666), np.float64(1.0)]
[0.         0.33333333 0.66666667 1.        ]


In [26]:
# Let's re-implement Z-scoring with Broadcasting.
# Z-score formula: z = (x-μ) / σ
# First we need a 2D dataset (subjects*features).

data = np.array([[1,2,3],[4,5,6],[7,8,9]])
mean = data.mean(axis=0)  # mean across rows (features)
std = data.std(axis=0)
z = (data - mean) / std

print(z)

[[-1.22474487 -1.22474487 -1.22474487]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  1.22474487]]
